In [4]:
import os

from dotenv import load_dotenv
from huggingface_hub import CommitOperationDelete, HfApi

load_dotenv()  # reads .env file, populates os.environ

True

In [5]:
is_hf_token_present = True if os.getenv("HF_TOKEN") else False
is_hf_token_present

True

In [6]:
api = HfApi(
    token=os.getenv("HF_TOKEN")
)  # picks up HF_TOKEN from env automatically if you ran load_dotenv() earlier
REPO_ID = "pfaha/goodreads-books"
REPO_TYPE = "dataset"
RAW_DIR = "raw"

all_files = api.list_repo_files(repo_id=REPO_ID, repo_type=REPO_TYPE)

parquet_files = [
    f
    for f in all_files
    if (
        (f.startswith(f"{RAW_DIR}/books-part") and f.endswith(".parquet"))
        or (f.startswith(f"{RAW_DIR}/book_ids") and f.endswith(".parquet"))
    )
]

print(f"Found {len(parquet_files)} file(s) to delete:")
for f in parquet_files:
    print(f"  {f}")

Found 3 file(s) to delete:
  raw/book_ids.parquet
  raw/books-part1.parquet
  raw/books-part2.parquet


In [7]:
operations = [CommitOperationDelete(path_in_repo=f) for f in parquet_files]

if operations:
    api.create_commit(
        repo_id=REPO_ID,
        repo_type=REPO_TYPE,
        operations=operations,
        commit_message=f"Delete {len(parquet_files)} old checkpoint files (schema cleanup)",
    )
    print(f"Deleted {len(parquet_files)} file(s)")
else:
    print("No matching files found -- nothing deleted")

Deleted 3 file(s)
